In [1]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 37.0 MB/s eta 0:00:00


In [3]:
import csv
import json
import time

import pandas as pd
import requests
import os
import logging
import sys
import json
import contextlib

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException


import logging
import sys
import os

def init(output_dir='data_bonbanh'):
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler("scraper_per_links_bonbanh.log", encoding='utf-8'),
            logging.StreamHandler(sys.stdout)
        ]
    )

    # BƯỚC 2: Thêm một mẹo để xử lý lỗi khi tạo thư mục
    try:
        os.makedirs(output_dir, exist_ok=True)
        for subdir in ["raw", "processed"]:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
    except Exception as e:
        print(f"Lỗi nghiêm trọng khi tạo thư mục: {str(e)}", file=sys.stderr)
        sys.exit(1)

    logging.info("Hệ thống logging và cấu trúc thư mục đã được khởi tạo thành công.")

@contextlib.contextmanager
def get_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        driver.set_page_load_timeout(50)
        yield driver
    finally:
        if driver:
            driver.quit()

def checkAcceptable(url_to_try, driver): # 👈 THÊM DRIVER VÀO ĐỐI SỐ
    headers = {
        # ... (giữ nguyên headers) ...
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'Accept-Language': 'vi,en-US;q=0.9,en;q=0.8',
        'Connection': 'keep-alive',
        'Referer': 'https://bonbanh.com/',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-User': '?1',
        'Upgrade-Insecure-Requests': '1',
        'Cache-Control': 'max-age=0',
    }
    session = requests.Session()
    try:
        response = session.get(url_to_try, timeout=50, headers=headers)
        if response.status_code != 200:
            logging.error(f"     Không thể kết nối đến {url_to_try} : Mã trạng thái {response.status_code}")
            return 0
    except requests.exceptions.RequestException as e:
        logging.error(f"     Không thể kết nối đến {url_to_try} : {str(e)}")
        return 0
    finally:
        if session is not None:
            session.close()

    # KHÔNG CÒN KHỞI TẠO OPTIONS VÀ DRIVER Ở ĐÂY NỮA
    try:
        driver.get(url_to_try) # 👈 TÁI SỬ DỤNG DRIVER

        if driver.current_url != url_to_try:
            logging.warning(f"     {url_to_try} sai lệch nên web đã trở về trang chủ https://bonbanh.com ")
            return 0

        wait = WebDriverWait(driver, 50)
        xpath_trigger_1 = "//div[@id='car_detail']//div[@class='title']"
        xpath_trigger_2 = "//div[@id='car_detail']//div[@id='sgg']"

        wait.until(EC.all_of(
            EC.presence_of_element_located((By.XPATH, xpath_trigger_1)),
            EC.presence_of_element_located((By.XPATH, xpath_trigger_2)),
        ))

        logging.info(f"     Truy cập thành công {url_to_try} !")
        return 1
    except Exception as ex:
        import traceback
        logging.warning(f"     Lỗi do thông tin không đầy đủ khi truy cập {url_to_try}: {type(ex).__name__} - {str(ex)}")
        logging.debug(traceback.format_exc())
        return 0
    # KHÔNG CÓ finally: driver.quit() Ở ĐÂY


def crawl_per_car(list_of_links):
    infos = []

    # 👈 KHỞI TẠO DRIVER MỘT LẦN DUY NHẤT CHO TOÀN BỘ DANH SÁCH LINKS
    with get_driver() as driver:
        for link in list_of_links:
            url_to_try = link
            logging.info(f"Đang thử truy cập {url_to_try}")

            # TRUYỀN DRIVER VÀO HÀM KIỂM TRA
            status = checkAcceptable(url_to_try, driver)
            logging.info(f"     Status là {status}")

            # Thêm time.sleep(5) sau khi checkAcceptable kết thúc
            # LƯU Ý: time.sleep(5) ở đây là để chống anti-bot, KHÔNG phải để fix SessionNotCreatedException.
            time.sleep(2)

            if status == 0:
                continue

            try:
                # Driver đã ở đúng trang do checkAcceptable() vừa gọi driver.get()

                # Chờ các thành phần trích xuất (bao gồm thông tin liên hệ bị thiếu trong checkAcceptable)
                wait = WebDriverWait(driver, 50)
                css_trigger_1 = "div#car_detail > div.title"
                css_trigger_2 = "div#car_detail > div#sgg"
                css_trigger_3 = "div#car_detail > div.contact-box > div.cinfo > div.contact-txt"
                wait.until(EC.all_of(
                    EC.presence_of_element_located((By.CSS_SELECTOR,css_trigger_1)),
                    EC.presence_of_element_located((By.CSS_SELECTOR, css_trigger_2)),
                    EC.presence_of_element_located((By.CSS_SELECTOR, css_trigger_3))
                ))

                css_selector_name = "div#car_detail > div.title > h1"
                css_selector_sell_time = "div#car_detail > div.title > div.notes"
                css_selector_info = "div#car_detail > div#sgg"
                css_selector_contact = "div#car_detail > div.contact-box > div.cinfo > div.contact-txt"

                info_per_car = extract_each_attributes(css_selector_name, css_selector_sell_time, css_selector_info, css_selector_contact, driver)
                infos.append(info_per_car)
            except Exception as e:
                logging.warning(f"     Lỗi không xác định khi trích xuất thông từ {url_to_try} - Lỗi chi tiết: {str(e)}")

            # Đã loại bỏ driver.quit() ở đây

        # driver.quit() sẽ được gọi tự động khi thoát khỏi khối with

    save_links_to_csv(infos)


def safe_find_text(driver, selector):
    """Hàm hỗ trợ: tìm phần tử, nếu không có thì trả về None"""
    try:
        return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
    except NoSuchElementException:
        return None


def extract_each_attributes(css_selector_name, css_selector_sell_time, css_selector_info, css_selector_contact, driver):
    logging.info(f"    Bắt đầu lấy các thông tin của xe {driver.current_url}")

    dct = dict.fromkeys([
        'name', 'sell_time', 'produce_year', 'current_status', 'traveled_kilometer', 'origin',
        'shape', 'gear_box', 'engine', 'out_color', 'in_color', 'seat_number', 'door_number', 'dynamic_axes',
        'describe_info', 'owner_name', 'phone_number_1', 'phone_number_2', 'owner_address'
    ])

    # Tên xe và thời gian đăng bán
    dct['name'] = safe_find_text(driver, css_selector_name)
    dct['sell_time'] = safe_find_text(driver, css_selector_sell_time)

    # Map nhãn (đã chuyển thường hết)
    mapping = {
        "năm sản xuất": "produce_year",
        "tình trạng": "current_status",
        "số km đã đi": "traveled_kilometer",
        "xuất xứ": "origin",
        "kiểu dáng": "shape",
        "hộp số": "gear_box",
        "động cơ": "engine",
        "màu ngoại thất": "out_color",
        "màu nội thất": "in_color",
        "số chỗ ngồi": "seat_number",
        "số cửa": "door_number",
        "dẫn động": "dynamic_axes"
    }

    try:
        info = driver.find_element(By.CSS_SELECTOR, css_selector_info)
        rows = info.find_elements(By.CSS_SELECTOR, "div.row, div.row_last")

        for row in rows:
            try:
                label_el = row.find_element(By.CSS_SELECTOR, "div.label")
                label = label_el.text.strip().replace(":", "").lower().strip()
                try:
                    # Trường hợp có <span class="inp">
                    value_el = row.find_element(By.CSS_SELECTOR, "div.txt_input > span.inp, div.inputbox > span.inp")
                    value = value_el.text.strip()
                except NoSuchElementException:
                    # Không có span, lấy text trực tiếp từ div.txt_input
                    try:
                        box_el = row.find_element(By.CSS_SELECTOR, "div.txt_input, div.inputbox")
                        value = box_el.text.strip()
                    except NoSuchElementException:
                        value = None

                if label in mapping:
                    dct[mapping[label]] = value if value else None
            except Exception:
                continue
    except NoSuchElementException:
        pass
    # Đảm bảo đủ 12 key
    for key in mapping.values():
        dct[key] = dct.get(key, None)

    # Mô tả
    try:
        dct['describe_info'] = info.find_element(By.CSS_SELECTOR,
                                                 "div.car_des > div.des_txt").text.strip() if info else None
    except NoSuchElementException:
        dct['describe_info'] = None

    # Thông tin liên hệ
    try:
        contact = driver.find_element(By.CSS_SELECTOR, css_selector_contact)
    except NoSuchElementException:
        contact = None

    # Tên người bán (a.cname hoặc span.cname)
    try:
        owner_elem = contact.find_elements(By.CSS_SELECTOR, "a.cname, span.cname") if contact else []
        dct['owner_name'] = owner_elem[0].text.strip() if owner_elem else None
    except Exception:
        dct['owner_name'] = None

    # Số điện thoại
    try:
        phones = contact.find_elements(By.CSS_SELECTOR, "span.cphone > a.cphone") if contact else []
        phone_texts = [p.text.strip() for p in phones if p.text.strip()]
        dct['phone_number_2'], dct['phone_number_1'] = ([None, None] + phone_texts)[-2:]
    except Exception:
        dct['phone_number_2'], dct['phone_number_1'] = None, None

    # Địa chỉ
    try:
        if contact:
            address_lines = [line.strip() for line in contact.text.split('\n') if line.strip()]
            found_address = None
            for line in address_lines:
                lower_line = line.lower()
                if "địa chỉ" in lower_line or "dia chi" in lower_line:
                    found_address = line.strip()
                    break
            dct['owner_address'] = found_address
        else:
            dct['owner_address'] = None

    except Exception as e:
        logging.debug(f"Lỗi khi lấy địa chỉ: {e}")
        dct['owner_address'] = None

    print(json.dumps(dct, ensure_ascii=False))#''', indent=2'''))
    return dct


def save_links_to_csv(infos, filename="info_cars.csv", output_dir='data_bonbanh'):
    try:
        os.makedirs(output_dir, exist_ok=True)
        file_path = os.path.join(output_dir,'raw', filename)
        file_exists = os.path.exists(file_path)

        with open(file_path, 'a+', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['name','sell_time','produce_year','current_status','traveled_kilometer','origin',
                          'shape','gear_box','engine','out_color','in_color','seat_number','door_number','dynamic_axes',
                          'describe_info','owner_name','phone_number_1','phone_number_2','owner_address']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()  # chỉ ghi header nếu file mới
            for info in infos:
                writer.writerow(info)
        logging.info(f"Đã lưu thêm {len(infos)} liên kết vào file: {file_path}")
    except Exception as e:
        logging.error(f"Lỗi khi lưu vào file CSV: {str(e)}")

# if __name__ == '__main__':
#     init()
#     lst_links = ['https://bonbanh.com/xe-mg-hs-1.5t-del-2025-6457197', 'https://bonbanh.com/xe-bmw-5_series-530i-m-sport-2022-6190459', 'https://bonbanh.com/xe-toyota-prado-vx-2.7l-2021-6367137']
#     crawl_per_car(lst_links)

def load_urls_from_csv(csv_path):
    """Đọc file CSV chứa danh sách URL."""
    if not os.path.exists(csv_path):
        logging.error(f"❌ File {csv_path} không tồn tại!")
        return []

    try:
        df = pd.read_csv(csv_path)
        if 'url' not in df.columns:
            logging.error("❌ File CSV không có cột 'url'!")
            return []
        urls = df['url'].dropna().tolist()
        logging.info(f"✅ Đọc được {len(urls)} URL từ {csv_path}")
        return urls
    except Exception as e:
        logging.error(f"❌ Lỗi khi đọc file CSV {csv_path}: {e}")
        return []

def main():
    init()
    csv_path = os.path.join("data_bonbanh", "sell_cars.csv")

    lst_links = load_urls_from_csv(csv_path)
    if not lst_links:
        logging.warning("⚠️ Không có URL nào để crawl.")
        return

    for i in range(2000,2500,10):
        logging.info(f"ĐANG CẠO LINK ROW {i} đến {i+10}")
        crawl_per_car(lst_links[i:i+10])  # Gọi hàm crawl như cũ

if __name__ == "__main__":
    main()


2025-10-06 02:04:01,461 - INFO - Hệ thống logging và cấu trúc thư mục đã được khởi tạo thành công.
2025-10-06 02:04:01,479 - INFO - ✅ Đọc được 5835 URL từ data_bonbanh/sell_cars.csv
2025-10-06 02:04:01,480 - INFO - ĐANG CẠO LINK ROW 2000 đến 2010
2025-10-06 02:04:12,030 - INFO - Đang thử truy cập https://bonbanh.com/xe-hyundai-tucson-2.0-at-dac-biet-2021-6454368
2025-10-06 02:04:23,859 - INFO -      Truy cập thành công https://bonbanh.com/xe-hyundai-tucson-2.0-at-dac-biet-2021-6454368 !
2025-10-06 02:04:23,860 - INFO -      Status là 1
2025-10-06 02:04:25,922 - INFO -     Bắt đầu lấy các thông tin của xe https://bonbanh.com/xe-hyundai-tucson-2.0-at-dac-biet-2021-6454368
{"name": "Xe Hyundai Tucson 2.0 AT Đặc biệt 2021 - 769 Triệu", "sell_time": "Đăng ngày 4/10/2025 . Xem 16 lượt", "produce_year": "2021", "current_status": "Xe đã dùng", "traveled_kilometer": "34,000 Km", "origin": "Lắp ráp trong nước", "shape": "Crossover", "gear_box": "Số tự động", "engine": "Xăng 2.0 L", "out_color": 